In [13]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_isx"


In [14]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values(["variable", "firm_id"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
ICEAIR.IC,COGS_DA,Income Statement,Depreciation and amortization,True,"Selected the summary D&A row because D&A is presented as a separate operating line, not as part of the already selected COGS component rows. Manual review is needed because the excerpt does not explicitly split D&A between direct aviation costs and SG&A/overhead, so assigning the full separate D&A line to COGS may overstate COGS-related D&A."
BERAHF.IC,XSGA_DA,Income Statement,Depreciation,True,"For the newer period, the 39.19 'Depreciation & Amortization' line looks like a component already included within selected SG&A row 'Other Operating Expense', so it should not be added. In the older periods, the separate same-level 'Depreciation' rows appear outside the selected SG&A rows ('Labour Expenses' and 'Other Operating Expense'), so D&A should be added separately. Manual review is needed because there are two distinct rows with the exact same label 'Depreciation', which cannot be uniquely identified by label alone and may create ambiguity."
EIKF.IC,XSGA_DA,Income Statement,Depreciation & Amortization,False,"A separate operating D&A line is visible before finance items and appears outside the selected SG&A row(s). No separate R&D row is visible, so there is no XRD offset concern here. Use only the summary D&A row, not its components."
EIMS.IC,XSGA_DA,Income Statement,Depreciation and amortization,False,"The selected XSGA_COMPONENTS bucket is 'Operatingexpenses' plus 'Salaries and related expenses'. The separate 'Depreciation and amortization' row appears outside that bucket at the same operating level, and in at least one year those rows sum to 'Operating Expense', confirming D&A is separate rather than embedded. Use only the summary row to avoid double counting with 'Depreciation' and 'Amortization of Intangibles' components."
FESTI.IC,XSGA_DA,Income Statement,Depreciation and amortization,False,"The earlier XSGA mapping intentionally avoided the parent row 'Operating Expense' because it clearly includes depreciation/amortization. Since 'Depreciation and amortization' appears as its own separate operating line outside the selected XSGA component rows, it should be added separately. Component rows are excluded because the summary row is available."
HAGA.IC,XSGA_DA,Income Statement,Depreciation/Amortization,False,"The selected XSGA_COMPONENTS rows are 'Salaries and related expenses' and 'Other Operating Expenses'. 'Depreciation/Amortization' is presented as a separate same-level operating line, so it is not already embedded in the selected XSGA bucket and should be added separately. No R&D row is selected, so the special R&D exclusion does not apply."
HAPD.IC,XSGA_DA,Income Statement,Depreciation/Amortization in SGA,False,"Add only the SGA D&A summary row. The selected XSGA bucket uses component rows ('Payroll Expenses' and 'Selling and Marketing Expense') rather than the total operating expense line, so the explicit separate SGA D&A line should be added later. XRD is empty, so there is no separate R&D row interaction to resolve."
HEIMAR.IC,XSGA_DA,Income Statement,Depreciation,False,Use the separate operating-level 'Depreciation' row for XSGA_DA because it appears distinct from the selected SG&A component rows and there is no more specific summary/component split to prefer.
ICESEA.IC,XSGA_DA,Income Statement,Depreciation and amortisation,False,"The selected XSGA bucket is 'Operating expenses'. The statement separately presents 'Depreciation and amortisation' above EBIT and the operating subtotal reconciles as though this D&A sits outside/alongside 'Operating expenses', so it should be added separately. Summary row selected to avoid double counting its components."
NOVA.IC,XSGA_DA,Income Statement,Depreciation and amortisation,False,"Use the summary D&A row only. It is a separate operating line above financial items and appears outside the selected XSGA component rows, so it should be added separ